# Maharashtra Data Preparation  --  v3
**ADB AI for Safer Roads Innovation Challenge 2026**

Reads the Maharashtra GeoJSON road-segment data and combines it with Helmet SPI data.
All SPI years (2022–2025 + `All` aggregate) are joined by LandUse.

| Step | What |
|---|---|
| 1 | Load GeoJSON |
| 2 | Load Helmet SPI Excel |
| 3 | Build SPI lookup tables |
| 4 | Combine datasets |
| 5 | Save outputs |

In [ ]:
import os, warnings, subprocess, sys
warnings.filterwarnings("ignore")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install",
     "geopandas", "pandas", "numpy", "openpyxl", "pyarrow"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import geopandas as gpd
import pandas as pd
import numpy as np
from IPython.display import display

SCRIPT_DIR  = os.path.dirname(os.path.abspath("__file__"))
BASE_DIR    = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(SCRIPT_DIR))))
GEOJSON     = os.path.join(BASE_DIR, "Data", "ADB_Innovation_Maharashtra.geojson")
HELMET_FILE = os.path.join(BASE_DIR, "Data", "Archive",
              "Road_Safety_Performance_Indicators_(Helmet_Wearing_results)_(adb_dashboard_data_v02).xlsx")
RESULTS_DIR = os.path.join(BASE_DIR, "results", "Analysis", "data preparation", "Maharashtra", "v3")
os.makedirs(RESULTS_DIR, exist_ok=True)

REGION = "Maharashtra"
print(f"Project root : {BASE_DIR}")
print(f"Results dir  : {RESULTS_DIR}")

## Step 1 — Load GeoJSON

In [ ]:
gdf = gpd.read_file(GEOJSON)
print(f"{len(gdf):,} road segments  |  {len(gdf.columns)} columns  |  CRS: {gdf.crs}")
display(gdf.drop(columns="geometry").head(3))

## Step 2 — Load Helmet SPI Excel

In [ ]:
helmet_raw = pd.read_excel(HELMET_FILE)

helmet_region = helmet_raw[helmet_raw["Location"] == REGION].copy()
all_years     = sorted(helmet_region["Year"].unique())
numeric_years = [y for y in all_years
                 if str(y).isdigit() or (isinstance(y, (int, float)) and not isinstance(y, bool))]

print(f"Years available : {all_years}")
print(f"Numeric years   : {numeric_years}")
display(helmet_region)

# Save full SPI table
spi_all = helmet_region.pivot_table(
    index=["Location", "LandUse", "Year"], columns="User", values="SPI"
).reset_index()
spi_all.columns.name = None
spi_all.to_csv(os.path.join(RESULTS_DIR, "spi_all_years.csv"), index=False)
print("[Saved] spi_all_years.csv")

## Step 3 — Build SPI Lookup Tables

- **All-year aggregate**: `HelmetSPI`, `HelmetSPI_Driver`, `HelmetSPI_Passenger`
- **Per-year**: `HelmetSPI_2022`, `HelmetSPI_2023`, `HelmetSPI_2024`, `HelmetSPI_2025`

In [ ]:
# All-year aggregate
helmet_pivot_all = (
    helmet_region[helmet_region["Year"] == "All"]
    .pivot_table(index=["Location", "LandUse"], columns="User", values="SPI")
    .reset_index()
)
helmet_pivot_all.columns.name = None
helmet_pivot_all = helmet_pivot_all.rename(columns={
    "All Riders": "HelmetSPI",
    "Driver":     "HelmetSPI_Driver",
    "Passenger":  "HelmetSPI_Passenger",
})
print("All-year SPI lookup:")
display(helmet_pivot_all)

# Per-year SPI (Combined level only — no Rural/Urban breakdown in Excel)
# Broadcast as a constant column onto every segment.
spi_year_cols = []
helmet_year_series = {}
if numeric_years:
    helmet_years = helmet_region[
        helmet_region["Year"].isin(numeric_years) &
        (helmet_region["User"] == "All Riders") &
        (helmet_region["LandUse"] == "Combined")
    ].copy()
    helmet_years["Year"] = helmet_years["Year"].astype(str)
    for _, row in helmet_years.iterrows():
        col = f"HelmetSPI_{row['Year']}"
        helmet_year_series[col] = row["SPI"]
        spi_year_cols.append(col)
    spi_year_cols = sorted(spi_year_cols)
    print(f"\nPer-year SPI (Combined, broadcast): {helmet_year_series}")

## Step 4 — Combine Datasets

Left-join both SPI tables onto the GeoJSON by `LandUse`.

**CompositeRisk = (PercentOverLimit / 100) × (1 − HelmetSPI)**

In [ ]:
gdf["LandUse_join"] = gdf["LandUse"].str.capitalize()
gdf["Location"]     = REGION

# Join 1: All-year aggregate SPI (by LandUse)
gdf = gdf.merge(
    helmet_pivot_all,
    left_on=["Location", "LandUse_join"],
    right_on=["Location", "LandUse"],
    how="left", suffixes=("", "_helmet"),
)
gdf.drop(columns=["LandUse_helmet"], errors="ignore", inplace=True)
gdf.drop(columns=["LandUse_join"], errors="ignore", inplace=True)

# Join 2: Per-year SPI — broadcast constant columns
for col, val in helmet_year_series.items():
    gdf[col] = val

# CompositeRisk scores
gdf["CompositeRisk"]           = (gdf["PercentOverLimit"] / 100) * (1 - gdf["HelmetSPI"])
gdf["CompositeRisk_Driver"]    = (gdf["PercentOverLimit"] / 100) * (1 - gdf["HelmetSPI_Driver"])
gdf["CompositeRisk_Passenger"] = (gdf["PercentOverLimit"] / 100) * (1 - gdf["HelmetSPI_Passenger"])

print(f"Combined: {len(gdf):,} rows, {len(gdf.columns)} columns")

spi_cols = ["HelmetSPI", "HelmetSPI_Driver", "HelmetSPI_Passenger",
            "CompositeRisk", "CompositeRisk_Passenger"] + spi_year_cols
display(pd.DataFrame([
    {"Column": c, "Non-null": gdf[c].notna().sum(),
     "Coverage %": f"{100*gdf[c].notna().mean():.1f}%"}
    for c in spi_cols if c in gdf.columns
]))

print("\nSample of combined data:")
display(gdf.drop(columns="geometry").head(5))

## Step 5 — Save Outputs

In [ ]:
gdf.drop(columns="geometry").to_csv(
    os.path.join(RESULTS_DIR, "maharashtra_combined.csv"), index=False
)
print("[Saved] maharashtra_combined.csv")

gdf.to_parquet(os.path.join(RESULTS_DIR, "maharashtra_combined.parquet"), index=False)
print("[Saved] maharashtra_combined.parquet  (geometry preserved)")

print(f"\nAll outputs saved to: {RESULTS_DIR}")